In [ ]:
# Cell 1: imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

%matplotlib inline
warnings.filterwarnings("ignore")

# preprocessing tools (kept for later use)
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# stats helpers if you want to run tests or outlier detection later
from scipy import stats
from scipy.stats import norm


In [ ]:
# Cell 2: load data and basic structure checks
df = pd.read_csv("./dataset-diabete.csv")

print("Shape:", df.shape)
display(df.head())
print("\nInfo:")
display(df.info())

# ---- Duplicate detection (add to Cell 2, after loading df) ----
dup_count = df.duplicated().sum()
print("Duplicate rows found:", dup_count)

# show a few duplicate rows (if any) for inspection
if dup_count > 0:
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(20))

df = df.drop_duplicates().reset_index(drop=True)
print("Dropped duplicates. New shape:", df.shape)

# Drop index-like column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    print("Dropped 'Unnamed: 0'")

print("\nDescriptive statistics (numeric cols):")
display(df.describe().T)



Shape: (768, 9)


,Unnamed: 0,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,0,6,148,72,35,0,33.6,0.627,50
1,1,1,85,66,29,0,26.6,0.351,31
2,2,8,183,64,0,0,23.3,0.672,32
3,3,1,89,66,23,94,28.1,0.167,21
4,4,0,137,40,35,168,43.1,2.288,33



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                768 non-null    int64  
 1   Pregnancies               768 non-null    int64  
 2   Glucose                   768 non-null    int64  
 3   BloodPressure             768 non-null    int64  
 4   SkinThickness             768 non-null    int64  
 5   Insulin                   768 non-null    int64  
 6   BMI                       768 non-null    float64
 7   DiabetesPedigreeFunction  768 non-null    float64
 8   Age                       768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


None

Duplicate rows found: 0
Dropped 'Unnamed: 0'

Descriptive statistics (numeric cols):


,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00


## Explanation of Cell 2: Load Data and Basic Structure Checks

### Purpose
This cell loads the diabetes dataset from a CSV file (`./dataset-diabete.csv`) and performs initial exploratory checks to understand its structure, data types, missing values, duplicates, and basic statistics. These steps are crucial for ensuring data quality and preparing for subsequent preprocessing, clustering, or modeling tasks.

### Methodology
- **Data Loading**: The dataset is read into a pandas DataFrame using `pd.read_csv`.
- **Basic Checks**:
  - **Shape**: Displays the number of rows and columns.
  - **Head**: Shows the first five rows to inspect raw data.
  - **Info**: Provides an overview of column names, non-null counts, and data types.
- **Duplicate Detection**:
  - Checks for duplicate rows using `duplicated()` and reports the count.
  - If duplicates exist, displays up to 20 duplicate rows for inspection (sorted for clarity).
  - A line allows dropping duplicates if deemed necessary.
- **Column Cleanup**: Removes the `Unnamed: 0` column (a common index-like artifact from CSV exports) if present.
- **Descriptive Statistics**: Generates summary statistics (count, mean, std, min, 25%, 50%, 75%, max) for numeric columns to assess distribution and potential outliers.

### Results
- **Shape**: (768, 9), indicating 768 rows (samples) and 9 columns (features).
- **Head**: 
  - Columns include `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`, and an initial `Unnamed: 0`.
  - Sample values show integers (e.g., `Pregnancies`, `Glucose`) and floats (e.g., `BMI`, `DiabetesPedigreeFunction`), with some zeros (e.g., `Insulin`, `SkinThickness`).
- **Info**:
  - All 768 entries are non-null across all columns.
  - Data types: 7 integer (`int64`), 2 float (`float64`), total memory usage 54.1 KB.
- **Duplicates**: 0 duplicate rows found, indicating no exact replicates in the dataset.
- **Column Cleanup**: `Unnamed: 0` was dropped, reducing the column count to 8.
- **Descriptive Statistics**:
  - **Pregnancies**: Mean = 3.85, Max = 17, suggesting a range from 0 to 17 pregnancies.
  - **Glucose**: Mean = 120.89, Max = 199, with zeros present (likely missing data).
  - **BloodPressure**: Mean = 69.11, Max = 122, with zeros indicating potential missingness.
  - **SkinThickness**: Mean = 20.54, Max = 99, with many zeros (likely missing).
  - **Insulin**: Mean = 79.80, Max = 846, with zeros suggesting missing values.
  - **BMI**: Mean = 31.99, Max = 67.1, with zeros (unrealistic, likely missing).
  - **DiabetesPedigreeFunction**: Mean = 0.47, Max = 2.42, a genetic risk score.
  - **Age**: Mean = 33.24, Max = 81, ranging from 21 to 81 years.
  - High standard deviations (e.g., `Insulin` = 115.24) and maximums (e.g., `BMI` = 67.1) hint at potential outliers.

### What This Tells Us
- **Data Integrity**: The absence of missing values and duplicates suggests the dataset is clean at a surface level, but zeros in clinical columns (e.g., `Glucose`, `Insulin`, `BMI`) are likely placeholders for missing data, a known issue in the Pima Indians Diabetes dataset.
- **Feature Characteristics**: All columns are numeric, with `Pregnancies`, `Age`, and `Glucose` as integers, and `BMI` and `DiabetesPedigreeFunction` as floats. The `Unnamed: 0` column was correctly identified as an index and removed.
- **Distribution Insights**: The wide ranges (e.g., `Insulin` 0–846, `BMI` 0–67.1) and high standard deviations indicate potential outliers or skewed distributions, which may require preprocessing (e.g., winsorization or imputation).
- **Sample Size**: 768 rows provide a reasonable dataset for modeling, though the presence of zeros suggests imputation will be necessary to handle missingness effectively.
- **Next Steps Implications**: The data is ready for further preprocessing (e.g., treating zeros as missing, scaling), clustering, or supervised modeling. The descriptive stats suggest focusing on outlier handling and feature engineering (e.g., log-transforming skewed columns like `Insulin`).

### Recommendations
- **Handle Zeros**: Treat zeros in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` as missing values (e.g., replace with `NaN`) in the next cell, followed by imputation (e.g., median) to align with clinical reality.
- **Outlier Check**: Investigate extreme values (e.g., `BMI` > 50, `Insulin` > 500) using boxplots or IQR in a follow-up cell to decide on capping or removal.
- **Proceed with Preprocessing**: Move to scaling and clustering (e.g., Cell 3) or supervised target definition, ensuring missing value handling is consistent across the pipeline.
- **Documentation**: Note the decision to drop `Unnamed: 0` and the absence of duplicates for transparency. Record observations (e.g., “Zeros in `Insulin` likely indicate missing data”) for future reference.

### Notes
- The dataset appears to be the Pima Indians Diabetes dataset, where zeros are a known proxy for missing values. Adjust interpretations if the dataset differs.
- If duplicates or missing values appear in later checks (e.g., after treating zeros), revisit this cell to drop duplicates or handle them accordingly.

In [ ]:
# === Cell 3 (updated): EDA + pairplot + IQR outlier detection + two remediation options ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# (Assumes df already exists and zeros->NaN marking was applied earlier)
# If you haven't replaced zeros with NaN yet, uncomment the next block:
# cols_zero_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
# for c in cols_zero_missing:
#     if c in df.columns:
#         df[c] = df[c].replace(0, np.nan)

# 1) Basic histograms (numeric)
numeric_df = df.select_dtypes(include=[np.number]).copy()
if numeric_df.shape[1] == 0:
    raise RuntimeError("No numeric columns found. Check dataset.")

plt.figure(figsize=(12, 10))
numeric_df.hist(figsize=(12,10))
plt.suptitle("Histograms (numeric features)", y=0.95)
plt.tight_layout()
plt.show()

# 2) Boxplots for two key variables
for col in ["Glucose", "BMI"]:
    if col in numeric_df.columns:
        plt.figure(figsize=(6,2.6))
        sns.boxplot(x=numeric_df[col])
        plt.title(f"Boxplot: {col}")
        plt.show()

# 3) Pairplot (sampled for speed)
pair_feats = ['Glucose', 'BMI', 'DiabetesPedigreeFunction', 'Age']
pair_feats = [c for c in pair_feats if c in df.columns]
sample = df[pair_feats].dropna().sample(min(300, len(df)), random_state=42)
sns.pairplot(sample, diag_kind='kde', plot_kws={'alpha':0.5, 's':20})
plt.suptitle("Pairplot (selected features) — sampled", y=1.02)
plt.show()

# 4) IQR outlier detection function and flags
def iqr_outlier_bounds(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k*iqr, q3 + k*iqr

numeric = df.select_dtypes(include=[np.number]).copy()
outlier_flags = pd.DataFrame(False, index=numeric.index, columns=numeric.columns)

for col in numeric.columns:
    # dropna when computing bounds so missing don't bias bounds
    lower, upper = iqr_outlier_bounds(numeric[col].dropna(), k=1.5)
    outlier_flags[col] = (numeric[col] < lower) | (numeric[col] > upper)

# summary counts
per_col_outliers = outlier_flags.sum().sort_values(ascending=False)
per_row_outliers = outlier_flags.sum(axis=1)

print("Outliers per column (IQR k=1.5):")
display(per_col_outliers[per_col_outliers > 0])

print("\nRows that have >=1 outlier:", int((per_row_outliers>0).sum()))
print("Rows with >=2 outlier features:", int((per_row_outliers>=2).sum()))

if (per_row_outliers>=2).sum() > 0:
    print("\nExample rows with >=2 outlier flags (first 10):")
    display(df.loc[per_row_outliers>=2].head(10))

# 5) Two remediation options: Winsorize (cap) OR drop rows with many outlier flags.
#    Choose one policy depending on domain / sample size / how aggressive you want to be.

# OPTION A: WINSORIZE / CAP using IQR bounds (recommended when you want to keep rows)
df_capped = df.copy()
numeric_cols = df_capped.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    lower, upper = iqr_outlier_bounds(df_capped[col].dropna(), k=1.5)
    df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)

# report effect of capping on counts beyond bounds
beyond_before = (numeric > numeric.quantile(0.75) + 1.5*(numeric.quantile(0.75)-numeric.quantile(0.25))).sum()
beyond_after = (df_capped.select_dtypes(include=[np.number]) > df_capped.select_dtypes(include=[np.number]).quantile(0.75) + 1.5*(df_capped.select_dtypes(include=[np.number]).quantile(0.75)-df_capped.select_dtypes(include=[np.number]).quantile(0.25))).sum()
print("\nOPTION A: Winsorize (capping) prepared as df_capped (not applied to main df).")

# OPTION B: DROP rows that have >= 2 outlier flags (aggressive, removes suspicious rows)
threshold_multi = 2
to_drop_idx = per_row_outliers[per_row_outliers >= threshold_multi].index
df_dropped = df.drop(index=to_drop_idx).reset_index(drop=True)
print(f"\nOPTION B: Dropped rows with >= {threshold_multi} outlier features: {len(to_drop_idx)} rows removed.")
print("New shape if OPTION B used:", df_dropped.shape)

# 6) Pick your downstream numeric_df: pick df (original), df_capped or df_dropped
# For reproducibility choose one. Here we keep df_capped as default cleaned version but do not overwrite original:
numeric_df_after = df_capped.select_dtypes(include=[np.number]).copy()

print("\nDefault downstream: numeric_df_after (winsorized/capped). If you prefer to drop rows, set numeric_df_after = df_dropped.select_dtypes(...).")

# Quick visual check after chosen policy
plt.figure(figsize=(12,8))
check_cols = [c for c in ['Glucose','BMI','DiabetesPedigreeFunction','Age'] if c in numeric_df_after.columns]
for i,col in enumerate(check_cols):
    plt.subplot(2,2,i+1)
    sns.histplot(numeric_df_after[col].dropna(), kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()

# Expose variables for downstream cells to use:
# - df (original), df_capped (winsorized), df_dropped (rows removed), numeric_df_after


### Cell 3 — EDA, Pairplot, Outlier Detection & Remediation options

What this cell does (step by step)
1. Recomputes histograms for all numeric features to inspect marginal distributions (skew, multi-modality).
2. Draws boxplots for the two clinically-important variables (Glucose, BMI) so we can visually spot extremes.
3. Uses a sampled pairplot (Glucose, BMI, DiabetesPedigreeFunction, Age) to inspect pairwise relationships and potential structure (useful before clustering).
4. Implements IQR-based outlier detection:
   - For each numeric column we compute the IQR bounds (Q1 - 1.5*IQR, Q3 + 1.5*IQR).
   - We flag values outside these bounds as outliers and aggregate counts per column and per row.
   - This gives us a measure of how many rows have one or multiple suspicious values.
5. Presents two reasonable remediation policies and prepares cleaned datasets for both:
   - **Option A (Winsorize / cap)**: clip values to the computed IQR bounds. This reduces the influence of extreme values while keeping all samples. This is a good default when sample size is limited or outliers are likely measurement extremes.
   - **Option B (Drop rows with multiple outliers)**: remove rows that have >= 2 outlier flags. This is a more aggressive policy and is useful when we trust that rows with multiple extreme values are corrupted or not representative.
6. The cell produces `df_capped` and `df_dropped` alongside the original `df`. The notebook defaults to `numeric_df_after = df_capped` for downstream steps — change it to `df_dropped` if you prefer dropping.

Why we do this
- The dataset (Pima) often uses zeros to mean "not recorded" in certain clinical columns — these should be marked NA before computing descriptive stats or imputing.
- Outliers can heavily influence clustering centroids and supervised models. Winsorizing reduces their leverage; dropping rows removes them altogether — choose based on domain context and how many rows would be removed.
- Because earlier outlier summary showed:
  - e.g. many outliers in `BloodPressure`, `Insulin`, `DiabetesPedigreeFunction` and 129 rows with >=1 outlier and 17 rows with >=2 outlier features,
  - we recommend inspecting those example rows (the cell prints them) and then either winsorizing or dropping a small set if they look erroneous.

How to choose between Option A and Option B
- Use **Option A (winsorize)** if:
  - You have relatively few rows that would be removed by Option B,
  - Or if domain experts think extreme values are real but you want to reduce influence.
- Use **Option B (drop rows)** if:
  - Rows with multiple outliers look like corrupted/erroneous measurements,
  - You can afford the reduction in sample size (check `df_dropped.shape`).
- Always keep the original `df` saved so you can re-run with the other policy and compare downstream model performance (compare silhouette, classification metrics, etc.).

Next steps after this cell
- Use `numeric_df_after` (or `df_dropped`) as your cleaned input for clustering in the next cell.
- Re-run your clustering elbow + silhouette plots to confirm improvements.
- If clustering or model metrics improve substantially after capping/dropping, document that change (include before/after metrics in your final report).


In [ ]:
# Cell 4: top absolute pairwise correlations (feature-feature)
numeric_df = df.select_dtypes(include=[np.number]).copy()

corr_mat = numeric_df.corr()
# absolute correlation matrix with diagonal masked
abs_corr = corr_mat.abs().where(~np.eye(len(corr_mat), dtype=bool))

# unstack and sort unique pairs (we'll remove duplicates A-B and B-A)
pairs = abs_corr.unstack().dropna().sort_values(ascending=False)
# reduce duplicates by keeping only one of each pair (A,B) vs (B,A)
pairs = pairs.reset_index()
pairs.columns = ['feature_1', 'feature_2', 'abs_corr']
# keep only feature_1 < feature_2 (alphabetical) to get unique unordered pairs
pairs_unique = pairs[pairs['feature_1'] < pairs['feature_2']].sort_values('abs_corr', ascending=False)

print("Top absolute feature-feature correlations (descending):")
display(pairs_unique.head(15).set_index(['feature_1','feature_2']))


### Cell 4 — Correlation summary and decision

**What we found**
- The strongest pairwise absolute correlation is `Age` vs `Pregnancies` ≈ **0.544** (moderate).
- Other correlations are moderate-to-low (0.18–0.44 range). No pair > 0.8.

**Decision**
- Because no very-high correlations exist, we **keep all numeric features** for clustering and tree-based supervised models. 
- For linear or regression-style models we will compute **VIF** and consider dropping or regularizing if VIF > 5.
- For clustering we will use the winsorized dataset (`df_capped`) by default to reduce the influence of extreme values (you can switch to `df_dropped` if you prefer removing suspicious rows).

**Next steps**
1. Run the VIF cell to check multicollinearity for linear models.  
2. Visualize top correlated pairs (scatter + regression) to see relationship shapes.  
3. Continue with clustering using `df_capped` (updated Cell 6) and then proceed to labeling clusters and supervised classification.


In [ ]:
# Compute VIF to quantify multicollinearity (use the numeric_df_after you prepared earlier)
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# choose dataframe to test VIF on - use your chosen cleaned numeric DF
vif_df_source = numeric_df_after.copy()   # numeric_df_after is set by your Cell 3 (winsorized by default)

# add constant (required by statsmodels)
X_vif = sm.add_constant(vif_df_source)

vif_data = []
for i, col in enumerate(X_vif.columns):
    if col == 'const':
        continue
    vif_val = variance_inflation_factor(X_vif.values, i)  # +1 because of const at position 0
    vif_data.append((col, float(vif_val)))

vif_df = pd.DataFrame(vif_data, columns=['feature','VIF']).sort_values('VIF', ascending=False).reset_index(drop=True)
print("VIF results (higher -> more multicollinearity):")
display(vif_df)

# plot the top N correlated pairs from your pairs_unique dataframe
top_n = 6
top_pairs = pairs_unique.head(top_n).reset_index()[['feature_1','feature_2','abs_corr']]

for _, r in top_pairs.iterrows():
    a, b = r['feature_1'], r['feature_2']
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=df_capped[a] if 'df_capped' in globals() else df[a],
                    y=df_capped[b] if 'df_capped' in globals() else df[b],
                    alpha=0.6)
    sns.regplot(x=df_capped[a] if 'df_capped' in globals() else df[a],
                y=df_capped[b] if 'df_capped' in globals() else df[b],
                scatter=False, color='red', line_kws={'linewidth':1})
    plt.title(f"{a} vs {b}  (|corr|={r['abs_corr']:.3f})")
    plt.xlabel(a); plt.ylabel(b)
    plt.tight_layout()
    plt.show()


## Explanation of Cell: Multicollinearity Assessment with VIF and Correlation Analysis

### Purpose
This cell evaluates **multicollinearity** among the numeric features in the dataset using the **Variance Inflation Factor (VIF)** and visualizes the relationships between the most correlated feature pairs through scatter plots with regression lines. Multicollinearity occurs when predictor variables are highly correlated, which can destabilize model coefficients and reduce interpretability in regression-based analyses. The VIF quantifies this issue, while the scatter plots provide a visual confirmation of correlations identified earlier (e.g., in `pairs_unique` from a prior correlation analysis).

### Methodology
- **VIF Computation**:
  - The analysis uses `numeric_df_after` (prepared in Cell 3, likely winsorized based on the previous capping step) as the source DataFrame.
  - A constant term is added to the data using `sm.add_constant` to fit an OLS regression model, which is required for VIF calculation.
  - The `variance_inflation_factor` function computes the VIF for each feature, where VIF = \( 1 / (1 - R^2) \) from regressing the feature against all others. Higher VIF values indicate greater multicollinearity.
  - Features with VIF > 5–10 are typically considered to have problematic multicollinearity, though this threshold can vary by context.
- **Correlation Visualization**:
  - The top 6 most correlated pairs (based on `abs_corr` from `pairs_unique`) are selected.
  - For each pair, a scatter plot is generated using the capped DataFrame (`df_capped`) if available, or the original `df`, with a red regression line overlaid to highlight the linear relationship.
  - The absolute correlation coefficient (`|corr|`) is displayed in the title to quantify the strength of the relationship.

### Results
- **VIF Results**:
  - The VIF values range from 1.06 (`DiabetesPedigreeFunction`) to 1.64 (`Age`), with all features showing VIF < 2.
  - This indicates **low to no significant multicollinearity** among the predictors. For reference:
    - `Age`: 1.64
    - `SkinThickness`: 1.59
    - `Insulin`: 1.52
    - `Pregnancies`: 1.44
    - `BMI`: 1.33
    - `Glucose`: 1.31
    - `BloodPressure`: 1.22
    - `DiabetesPedigreeFunction`: 1.06
  - Since all VIF values are well below the common threshold of 5, the features are not strongly linearly dependent on each other.
- **Scatter Plots**:
  - The plots for the top 6 correlated pairs (e.g., potentially including `Glucose` vs. `BMI`, `Age` vs. `Pregnancies`, etc., depending on `pairs_unique`) show the data points and regression lines.
  - The `|corr|` values in the titles indicate the strength of linear relationships, with higher values suggesting tighter clustering around the regression line.
  - The use of `df_capped` (if applied) ensures that outliers have been mitigated, making the correlations and trends more reliable.

### What This Tells Us
- **Multicollinearity Assessment**: The low VIF values (all < 2) suggest that the numeric features (`Age`, `SkinThickness`, `Insulin`, etc.) are not highly collinear. This is a positive finding, as it implies that each feature contributes unique information to a regression model, reducing the risk of unstable coefficients or overfitting due to redundancy.
- **Feature Relationships**: The scatter plots confirm the correlations identified earlier. For example, if `Glucose` and `BMI` are among the top pairs with a high `|corr|` (e.g., > 0.5), the plots would show a clear trend, supported by the regression line. This visual validation helps understand how features interact, which is crucial for feature selection or interpreting model results.
- **Data Quality**: The use of `numeric_df_after` (winsorized in Cell 3) ensures that the VIF analysis is based on cleaned data, minimizing the influence of extreme values that could artificially inflate multicollinearity.
- **Modeling Implications**: With no significant multicollinearity, you can proceed with modeling (e.g., regression or machine learning) using all features without needing to drop or transform them to address collinearity. However, the scatter plots may reveal non-linear relationships or clusters that could warrant further investigation (e.g., using polynomial features or clustering).

### Recommendations
- **Interpret VIF**: The low VIF values are encouraging, but if you plan to use a regression model, consider a threshold of 5 as a conservative cutoff. Since all values are below 2, no action is needed unless specific domain knowledge suggests otherwise (e.g., `Age` and `Pregnancies` might have a biological relationship worth exploring).
- **Review Scatter Plots**: Examine the top 6 plots closely. If any pair shows a very high `|corr|` (e.g., > 0.7) with a tight regression line, consider whether one feature might be redundant, even with low VIF, especially in a predictive context.
- **Next Steps**: Proceed with modeling using the current feature set, confident that multicollinearity is not a concern. If scatter plots reveal non-linear trends, you might explore feature engineering (e.g., interactions or polynomials) in later cells.
- **Documentation**: Note any observations from the scatter plots (e.g., “Strong linear trend between Glucose and BMI with |corr|=0.65”) to inform future analysis or model interpretation.

### Notes
- The exact feature pairs in the scatter plots depend on the `pairs_unique` DataFrame, which should be generated in a prior cell (e.g., from a correlation matrix). Ensure this DataFrame is correctly populated.
- If `numeric_df_after` or `df_capped` is not defined as expected, check Cell 3’s output or the capping decision in the previous cell.

In [ ]:
## cell 5
# ---- Option A: Cap (winsorize) using IQR bounds (recommended) ----
df_capped = df.copy()
numeric_cols = df_capped.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    lower, upper = iqr_outlier_bounds(df_capped[col].dropna(), k=1.5)
    df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)

# quick compare before/after with boxplots (example for two key cols)
for col in ['Glucose','BMI']:
    if col in numeric_cols:
        plt.figure(figsize=(8,3))
        plt.subplot(1,2,1)
        sns.boxplot(x=df[col].dropna())
        plt.title(f"Original {col}")
        plt.subplot(1,2,2)
        sns.boxplot(x=df_capped[col].dropna())
        plt.title(f"Capped {col}")
        plt.tight_layout()
        plt.show()

# If satisfied, you can replace df with df_capped for further steps:
# df = df_capped.copy()

# ---- Quick check distributions after policy ----
numeric_after = (df if 'df' in globals() else df_capped).select_dtypes(include=[np.number])
plt.figure(figsize=(12,8))
for i,col in enumerate(['Glucose','BMI','DiabetesPedigreeFunction','Age']):
    if col in numeric_after.columns:
        plt.subplot(2,2,i+1)
        sns.histplot(numeric_after[col].dropna(), kde=True)
        plt.title(col)
plt.tight_layout()
plt.show()

# If you are satisfied with capping results, replace original df for downstream work:
use_capping = True   # set to True after you inspected the before/after plots
if use_capping:
    df = df_capped.copy()
    print("Applied capping: df replaced with df_capped (used for clustering and later steps).")
else:
    print("Capping NOT applied; using original df.")



## Explanation of Cell 5: Outlier Capping Using IQR Bounds

### Purpose
This cell implements a data preprocessing step to handle outliers in the dataset by applying a **winsorization** technique, which caps extreme values at the boundaries defined by the Interquartile Range (IQR) method. Outliers can skew statistical models and affect the performance of machine learning algorithms, so this step aims to mitigate their impact while preserving the overall distribution of the data. The process focuses on numeric columns (e.g., `Glucose`, `BMI`, `DiabetesPedigreeFunction`, `Age`) and uses a multiplier (`k=1.5`) to determine the capping thresholds.

### Methodology
- **IQR Bounds Calculation**: For each numeric column, the IQR is calculated as the difference between the 75th percentile (Q3) and the 25th percentile (Q1). The lower and upper bounds are set as:
  - Lower bound = Q1 - \( k \times \text{IQR} \)
  - Upper bound = Q3 + \( k \times \text{IQR} \), where \( k = 1.5 \) (a common choice for moderate outlier handling).
- **Winsorization**: Values below the lower bound are set to the lower bound, and values above the upper bound are set to the upper bound, effectively "capping" the outliers.
- **Visualization**: Before-and-after boxplots are generated for key columns (`Glucose` and `BMI`) to visually compare the original and capped distributions, helping to assess the effectiveness of the capping. Additionally, histograms with kernel density estimates (KDE) are plotted for all numeric columns to check the overall distribution post-capping.
- **Decision Point**: A boolean flag (`use_capping`) allows the user to decide whether to replace the original DataFrame (`df`) with the capped version (`df_capped`) based on the visual inspection. If `use_capping` is `True`, the capped data is used for subsequent steps (e.g., clustering or modeling).

### Results
- **Boxplots**: The side-by-side boxplots for `Glucose` and `BMI` show a reduction in the length of the whiskers and fewer (or no) extreme outliers beyond the IQR bounds after capping. This indicates that extreme values have been constrained, potentially leading to a more stable dataset for modeling.
- **Histograms**: The histograms for `Glucose`, `BMI`, `DiabetesPedigreeFunction`, and `Age` reveal the distribution shapes after capping. You should observe that the tails of the distributions are truncated compared to the original data, with the KDE curves smoothing out where extreme values were previously present. This suggests that the capping has reduced the influence of outliers without drastically altering the central tendency or overall shape.
- **Decision Outcome**: If `use_capping` is set to `True` (after inspection), a message confirms that `df` has been replaced with `df_capped`, ensuring downstream analyses (e.g., clustering or predictive modeling) use the cleaned data. If set to `False`, the original `df` is retained.

### What This Tells Us
- **Outlier Impact**: The presence of outliers in the original data (evident from the extended whiskers in boxplots) could have skewed model performance, especially for algorithms sensitive to extreme values (e.g., linear regression or SVM). Capping these outliers helps create a more robust dataset.
- **Effectiveness of Capping**: The reduced spread in boxplots and the adjusted histograms indicate that the IQR-based winsorization successfully mitigates extreme values while preserving the majority of the data's structure. This is particularly important for features like `BMI` and `Glucose`, which may contain health-related outliers (e.g., unusually high glucose levels).
- **Decision Flexibility**: The conditional replacement of `df` with `df_capped` allows for manual validation. If the capping appears to distort the data (e.g., overly aggressive truncation), you can opt to retain the original `df`, ensuring the preprocessing step is tailored to the specific needs of your analysis.
- **Next Steps**: With capping applied (if chosen), the cleaned dataset is better suited for clustering or predictive modeling, potentially improving model accuracy and stability. The visual tools provide a quick way to validate this step before proceeding.

### Recommendations
- **Inspect Plots**: Carefully review the before/after boxplots and histograms. If the capping removes too many data points or alters the distribution unnaturally (e.g., flattening peaks), consider adjusting `k` (e.g., to 3.0 for less aggressive capping) or exploring alternative methods (e.g., z-score clipping).
- **Proceed with Caution**: If satisfied with the results, set `use_capping = True` to apply the changes. Otherwise, keep it `False` and document why the original data is preferred for your specific use case (e.g., clustering might benefit from retaining natural variability).
- **Documentation**: Note any observations from the plots (e.g., significant outlier reduction in `BMI`) in your notebook for future reference or collaboration.

In [ ]:
# # === small reproducibility & integration fixes ===

# # 1) Ensure we have a canonical "numeric_df_after" used by later cells (VIF, correlations, clustering).
# numeric_df_after = df_capped.select_dtypes(include=[np.number]).copy()
# print("numeric_df_after shape:", numeric_df_after.shape)

# # 2) If you want to optionally remove rows that have many flagged outliers instead of capping:
# #    (For example: drop rows that had >= 2 IQR-outlier flags as identified earlier)
# drop_rows_with_many_outliers = False   # change to True if you want to drop them
# if drop_rows_with_many_outliers:
#     # per_row_outliers computed earlier in Cell 3; recompute here robustly if needed
#     _, per_row_outliers = None, None
#     # recompute flags on the capped-original dataframe (before capping would be better if you want to drop original outliers)
#     numeric_orig = df.select_dtypes(include=[np.number]).copy()  # use original df (before you swapped with df_capped)
#     outlier_flags_local = pd.DataFrame(False, index=numeric_orig.index, columns=numeric_orig.columns)
#     for col in numeric_orig.columns:
#         l,u = iqr_outlier_bounds(numeric_orig[col].dropna(), k=1.5)
#         outlier_flags_local[col] = (numeric_orig[col] < l) | (numeric_orig[col] > u)
#     per_row_outliers = outlier_flags_local.sum(axis=1)
#     rows_to_drop = per_row_outliers[per_row_outliers >= 2].index
#     print(f"Dropping {len(rows_to_drop)} rows that have >=2 IQR-outlier features.")
#     df_dropped = df.drop(index=rows_to_drop).reset_index(drop=True)
#     # update numeric_df_after and df if you choose dropping
#     numeric_df_after = df_dropped.select_dtypes(include=[np.number]).copy()
#     if drop_rows_with_many_outliers:
#         df = df_dropped.copy()
#         print("Applied drop policy: df replaced with df_dropped")

# # 3) If you applied capping, and set use_capping True, make sure df is updated (your code already does this):
# if use_capping:
#     df = df_capped.copy()

# # 4) Recompute pairs_unique (so correlation-based pairs refer to the updated numeric_df_after)
# corr_mat = numeric_df_after.corr()
# abs_corr = corr_mat.abs().where(~np.eye(len(corr_mat), dtype=bool))
# pairs = abs_corr.unstack().dropna().sort_values(ascending=False).reset_index()
# pairs.columns = ['feature_1', 'feature_2', 'abs_corr']
# pairs_unique = pairs[pairs['feature_1'] < pairs['feature_2']].sort_values('abs_corr', ascending=False)
# print("Updated top pairs (post-capping):")
# display(pairs_unique.head(10).set_index(['feature_1','feature_2']))

# # 5) Persist cleaned dataset (optional)
# # df_capped.to_csv("dataset_diabete_capped.csv", index=False)
# # print("Saved df_capped to dataset_diabete_capped.csv")


In [ ]:
# Cell : clustering prep + elbow & silhouette
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Work on a copy
df_proc = df.copy()

features = ["Glucose", "BMI", "DiabetesPedigreeFunction",
            "BloodPressure", "SkinThickness", "Insulin", "Age", "Pregnancies"]
# keep only features that exist in dataset
features = [c for c in features if c in df_proc.columns]
X = df_proc[features].copy()

# Treat zeros as missing for known columns (Pima convention)
cols_zero_missing = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
for c in cols_zero_missing:
    if c in X.columns:
        X[c] = X[c].replace(0, np.nan)

# Impute median (for clustering/EDA; for supervised pipeline we'll fit later properly)
X = X.fillna(X.median())

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Choose K range (start at 2)
K_vals = range(2, 8)
inertias = []
sil_scores = []

for k_ in K_vals:
    km_ = KMeans(n_clusters=k_, random_state=42, n_init=50)
    km_.fit(X_scaled)
    inertias.append(km_.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km_.labels_))

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(list(K_vals), inertias, '-o')
plt.xlabel("k (clusters)")
plt.ylabel("Inertia")
plt.title("Elbow plot")
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(list(K_vals), sil_scores, '-o')
plt.xlabel("k (clusters)")
plt.ylabel("Silhouette score")
plt.title("Silhouette scores")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 7 (replace any previous versions)
from sklearn.cluster import KMeans
import pandas as pd

def fit_and_summarize(k, X_scaled, scaler, X_orig):
    """
    Fit KMeans on X_scaled, return (km, labels, centers_df).
    X_orig is the original (unscaled) DataFrame used to name columns.
    """
    # defensive checks
    if X_scaled is None:
        raise RuntimeError("X_scaled is None. Ensure you computed X_scaled before calling this.")
    if scaler is None:
        raise RuntimeError("scaler is None. Provide a fitted scaler used to scale X_scaled.")
    if X_orig is None:
        raise RuntimeError("X_orig is None. Provide original DataFrame with column names.")

    km = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = km.fit_predict(X_scaled)
    centers = scaler.inverse_transform(km.cluster_centers_)
    centers_df = pd.DataFrame(centers, columns=X_orig.columns)
    centers_df['Cluster'] = range(k)
    centers_df = centers_df[['Cluster'] + list(X_orig.columns)]

    print(f"\n=== Summary for k={k} ===")
    display(centers_df.sort_values(by=centers_df.columns[1], ascending=False))  # sort by first feature (readable)
    counts = pd.Series(labels).value_counts().sort_index()
    print("Cluster counts:", counts.to_dict())
    return km, labels, centers_df

# Example quick run (only if X_scaled, scaler and X exist)
if 'X_scaled' in globals() and 'scaler' in globals() and 'X' in globals():
    km2, labels2, centers2 = fit_and_summarize(2, X_scaled, scaler, X)
    km3, labels3, centers3 = fit_and_summarize(3, X_scaled, scaler, X)
else:
    print("Note: X_scaled/scaler/X not found in globals. Run clustering prep cells first.")


In [ ]:
## Explanation of Cell: Clustering Preparation and Elbow/Silhouette Analysis

### Purpose
This cell prepares the dataset for unsupervised clustering analysis and evaluates the optimal number of clusters using the **elbow method** and **silhouette scores**. Clustering is used to identify natural groupings within the data (e.g., patient subgroups based on health metrics), which can provide insights for exploratory data analysis (EDA) or inform supervised modeling. The process involves data preprocessing (handling missing values, scaling) and the application of the K-means algorithm to assess clustering quality across different numbers of clusters (k).

### Methodology
- **Data Preparation**:
  - A copy of the DataFrame (`df_proc`) is created to avoid modifying the original data.
  - Selected features (`Glucose`, `BMI`, `DiabetesPedigreeFunction`, `BloodPressure`, `SkinThickness`, `Insulin`, `Age`, `Pregnancies`) are extracted, retaining only those present in the dataset.
  - Zeros in specific columns (`Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`) are treated as missing values (a convention in the Pima Indians Diabetes dataset), replaced with `NaN`.
  - Missing values are imputed with the median of each column, a simple strategy suitable for clustering and EDA (a more robust imputation will be applied later for supervised tasks).
  - Features are standardized using `StandardScaler` to ensure all variables contribute equally to the clustering, as K-means is sensitive to the scale of the data.
- **Clustering Analysis**:
  - The K-means algorithm is run for \( k \) values from 2 to 7 (inclusive), with 50 initializations (`n_init=50`) to improve convergence and reduce the risk of local minima, using a fixed random state for reproducibility.
  - **Elbow Plot**: Inertia (within-cluster sum of squares) is calculated for each \( k \), plotted against \( k \) to identify the "elbow" point where adding more clusters yields diminishing returns in reducing inertia.
  - **Silhouette Scores**: The silhouette score (ranging from -1 to 1) measures how similar an object is to its own cluster compared to other clusters, with higher values indicating better-defined clusters. This score is computed for each \( k \) to assess cluster cohesion and separation.
  - Both metrics are visualized side-by-side for comparison.

### Results
- **Elbow Plot**: The plot shows inertia decreasing as \( k \) increases from 2 to 7. The "elbow" (point of inflection) indicates the optimal \( k \) where the rate of inertia reduction slows significantly. For example, if the curve flattens around \( k = 3 \) or \( k = 4 \), this suggests 3 or 4 clusters might be appropriate.
- **Silhouette Scores**: The plot displays silhouette scores for each \( k \). A peak in the score (e.g., at \( k = 3 \) or \( k = 4 \)) indicates the number of clusters where objects are most cohesively grouped and well-separated from other clusters. Scores above 0.5 are generally considered good, while scores near 0 or negative suggest overlapping or poorly defined clusters.
- **Specific Observations**: The exact shape of the plots depends on the data. For instance, if the silhouette score peaks at \( k = 3 \) with a value > 0.5 and the elbow occurs around the same \( k \), it strongly supports 3 clusters.

### What This Tells Us
- **Clustering Feasibility**: The preprocessing steps (imputing zeros as missing and scaling) ensure the data is suitable for K-means. The treatment of zeros as `NaN` reflects domain knowledge (e.g., zero insulin or blood pressure is unrealistic in the Pima dataset), and median imputation provides a stable baseline for clustering.
- **Optimal Number of Clusters**: The elbow plot helps identify where adding more clusters provides little additional benefit, while the silhouette score validates this by measuring cluster quality. A clear elbow and high silhouette score at the same \( k \) (e.g., 3 or 4) suggest a natural grouping in the data, potentially reflecting distinct patient profiles (e.g., healthy, pre-diabetic, diabetic).
- **Data Structure**: If the silhouette score remains low (< 0.3) across all \( k \), it indicates that the data may not have well-defined clusters, possibly due to high variability or noise. Conversely, a high score (e.g., > 0.5) at an elbow point confirms strong clustering potential.
- **Modeling Implications**: The chosen \( k \) can guide subsequent analyses, such as assigning cluster labels for EDA or using cluster membership as a feature in supervised models. However, the simplicity of median imputation and the focus on EDA suggest this is an exploratory step, with more rigorous preprocessing planned for predictive modeling.

### Recommendations
- **Interpret Plots**: Examine the elbow plot for a clear bend (e.g., \( k = 3 \) or \( k = 4 \)) and the silhouette plot for the highest score. Select the \( k \) where both metrics align (e.g., if the elbow is at 3 and the silhouette peaks at 0.6, choose \( k = 3 \)).
- **Refine if Needed**: If the silhouette score is low or the elbow is unclear, consider:
  - Removing features with low variance or high missingness (e.g., `SkinThickness` or `Insulin` if sparsely populated).
  - Testing a wider \( k \) range (e.g., 2 to 10) or using alternative clustering methods (e.g., DBSCAN or hierarchical clustering).
- **Next Steps**: Once \( k \) is chosen, run K-means with that number of clusters, assign labels to `df_proc`, and explore cluster characteristics (e.g., mean values per cluster) in a subsequent cell. For supervised modeling, revisit imputation with a fitted transformer (e.g., `SimpleImputer`) on the full dataset.
- **Documentation**: Record the selected \( k \) and key observations (e.g., “Elbow at \( k = 3 \), silhouette score 0.55”) to justify the clustering decision.

### Notes
- The results depend on the specific dataset (e.g., Pima Indians Diabetes). If features like `SkinThickness` or `Insulin` have many zeros or missing values post-imputation, their impact on clustering should be validated.
- Ensure `pairs_unique` (from a prior correlation cell) and `numeric_df_after` (from Cell 3) are consistent with the features used here.

In [ ]:
# Cell 8 (replace previous)
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score, silhouette_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_and_inspect(k, X_scaled, scaler, X_orig, show_plots=True):
    # call fit_and_summarize (must be the 4-arg version)
    km, labels, centers_df = fit_and_summarize(k, X_scaled, scaler, X_orig)

    inertia = km.inertia_
    sil = silhouette_score(X_scaled, labels) if len(np.unique(labels))>1 else np.nan
    dbi = davies_bouldin_score(X_scaled, labels) if len(np.unique(labels))>1 else np.nan
    ch = calinski_harabasz_score(X_scaled, labels) if len(np.unique(labels))>1 else np.nan

    print(f"\nMetrics for k={k}: inertia={inertia:.2f}, silhouette={sil:.4f}, DBI={dbi:.4f}, CH={ch:.2f}")
    unique, counts = np.unique(labels, return_counts=True)
    counts_dict = dict(zip(unique, counts))
    print("Cluster distribution:", counts_dict)

    if show_plots:
        # PCA for 2D visualization
        pca = PCA(n_components=2, random_state=42)
        X_pca = pca.fit_transform(X_scaled)
        df_plot = pd.DataFrame(X_pca, columns=['PC1','PC2'])
        df_plot['cluster'] = labels

        plt.figure(figsize=(8,6))
        sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='cluster', palette='tab10', s=40, alpha=0.8)
        plt.title(f'PCA 2D view (k={k})')
        plt.legend(title='cluster', bbox_to_anchor=(1.05,1), loc='upper left')
        plt.tight_layout()
        plt.show()

        # show cluster centroids on PCA axes (optional)
        try:
            centers_scaled = km.cluster_centers_
            centers_pca = pca.transform(centers_scaled)
            plt.figure(figsize=(6,4))
            sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], alpha=0.3, s=20)
            plt.scatter(centers_pca[:,0], centers_pca[:,1], c='red', marker='X', s=100)
            plt.title(f"PCA plot with cluster centroids (k={k})")
            plt.show()
        except Exception:
            pass

    return {"k":k, "km":km, "labels":labels, "inertia":inertia, "silhouette":sil, "dbi":dbi, "ch":ch, "counts":counts_dict}

# Run diagnostics for k=2 and k=3 (if X_scaled/scaler/X available)
if 'X_scaled' in globals() and 'scaler' in globals() and 'X' in globals():
    res2 = evaluate_and_inspect(2, X_scaled, scaler, X, show_plots=True)
    res3 = evaluate_and_inspect(3, X_scaled, scaler, X, show_plots=True)
else:
    print("X_scaled, scaler, or X missing — run preprocessing/clustering preparation cells first.")


## Explanation of Cell 8: Clustering Evaluation with PCA and Metrics

### Purpose
This cell evaluates the quality of K-means clustering for \( k = 2 \) and \( k = 3 \) clusters using multiple metrics (inertia, silhouette score, Davies-Bouldin Index, and Calinski-Harabasz score) and visualizes the results in a 2D space using **Principal Component Analysis (PCA)**. The goal is to assess how well the data separates into natural groups based on the selected features (`Glucose`, `BMI`, `DiabetesPedigreeFunction`, etc.) and to determine the optimal number of clusters for further analysis or modeling. The `fit_and_summarize` function (assumed defined elsewhere) provides cluster centroids, while PCA reduces dimensionality for visualization.

### Methodology
- **Clustering Setup**:
  - The preprocessed and scaled data (`X_scaled`), scaler (`scaler`), and original data (`X`) from prior cells are used.
  - The `evaluate_and_inspect` function runs K-means with a specified \( k \), fits the model, and computes:
    - **Inertia**: The within-cluster sum of squares, lower values indicate tighter clusters.
    - **Silhouette Score**: Measures how similar an object is to its own cluster vs. other clusters (range: -1 to 1, higher is better).
    - **Davies-Bouldin Index (DBI)**: Measures the average similarity ratio of each cluster with its most similar cluster (lower is better, < 1 is good).
    - **Calinski-Harabasz Score (CH)**: Ratio of between-cluster dispersion to within-cluster dispersion (higher is better).
  - Cluster distributions and centroids (transformed back to original scale) are reported.
- **Visualization**:
  - PCA reduces the data to 2 components for 2D plotting.
  - Scatter plots show data points colored by cluster, with optional centroid markers to highlight cluster centers.
  - Plots are generated for both \( k = 2 \) and \( k = 3 \) to compare visual separation.

### Results
- **For \( k = 2 \)**:
  - **Centroids**: 
    - Cluster 0: Higher `Glucose` (137.55), `BMI` (34.97), `Age` (41.01), `Pregnancies` (5.67), suggesting older, potentially diabetic patients.
    - Cluster 1: Lower `Glucose` (106.81), `BMI` (29.64), `Age` (26.38), `Pregnancies` (2.23), indicating younger, healthier individuals.
  - **Cluster Counts**: 358 (Cluster 0), 410 (Cluster 1).
  - **Metrics**: Inertia = 4956.13, Silhouette = 0.1884, DBI = 1.9454, CH = 183.59.
  - **PCA Plot**: Two clusters are visible, with some overlap, and centroids (red 'X' markers) show separation along the first two principal components.
- **For \( k = 3 \)**:
  - **Centroids**: 
    - Cluster 0: High `BMI` (38.56), `Insulin` (172.04), low `Pregnancies` (1.90), possibly obese with insulin issues.
    - Cluster 1: Low `Glucose` (105.18), `BMI` (27.65), `Age` (26.08), healthy profile.
    - Cluster 2: High `Age` (45.82), `Pregnancies` (7.25), moderate `Glucose` (130.08), older multiparous patients.
  - **Cluster Counts**: 198 (Cluster 0), 318 (Cluster 1), 252 (Cluster 2).
  - **Metrics**: Inertia = 4314.11, Silhouette = 0.1865, DBI = 1.8417, CH = 162.24.
  - **PCA Plot**: Three clusters are visible, with increased separation but some overlap, and centroids indicate distinct centers.

### What This Tells Us
- **Cluster Separation**: The lower inertia for \( k = 3 \) (4314.11 vs. 4956.13) suggests tighter clusters, but the silhouette score drops slightly (0.1865 vs. 0.1884), indicating marginally worse cohesion and separation. The DBI improves (1.8417 vs. 1.9454), and CH decreases (162.24 vs. 183.59), reflecting a trade-off between cluster tightness and separation.
- **Cluster Interpretation**:
  - For \( k = 2 \), the split aligns with a healthy vs. at-risk dichotomy, with Cluster 0 representing older, higher-risk patients and Cluster 1 younger, healthier ones.
  - For \( k = 3 \), the additional cluster (Cluster 0) highlights a subgroup with high `BMI` and `Insulin`, possibly indicating obesity-related diabetes risk, while Cluster 2 captures older, multiparous patients.
- **Optimal \( k \)**: The elbow plot (from the previous cell) and silhouette scores suggest \( k = 2 \) or \( k = 3 \) as candidates. Here, \( k = 2 \) has a slightly better silhouette score (0.1884 vs. 0.1865), but \( k = 3 \) offers more granularity with a lower DBI (1.8417), indicating better-defined clusters. The PCA plots show moderate separation, with \( k = 3 \) revealing an additional subgroup.
- **Data Structure**: The overlap in PCA plots suggests the data has inherent variability, and the features may not perfectly separate into distinct clusters. The centroid differences (e.g., `BMI` and `Age`) align with clinical expectations, supporting the clustering’s validity.
- **Modeling Implications**: The clusters can be used as features in supervised models or for EDA (e.g., comparing diabetes prevalence across clusters). However, the low silhouette scores (< 0.2) indicate that the clusters are not very distinct, suggesting clustering may be exploratory rather than definitive.

### Recommendations
- **Choose \( k \)**: Select \( k = 3 \) if you prefer more detailed subgroups (e.g., for targeted health interventions), as it balances inertia, DBI, and clinical interpretability. Opt for \( k = 2 \) if a simpler healthy vs. at-risk split is sufficient, given the slightly better silhouette score.
- **Refine Clustering**: If silhouette scores remain low, consider:
  - Removing features with high missingness (e.g., `SkinThickness`, `Insulin`) or low variance.
  - Testing other algorithms (e.g., Gaussian Mixture Models) or adjusting \( k \) (e.g., 4 or 5).
- **Validate Clusters**: In a follow-up cell, analyze cluster characteristics (e.g., diabetes rates) to assess practical utility. Use the centroids and counts to guide interpretation.
- **Next Steps**: Assign the chosen \( k \)’s labels to `df_proc` and proceed with EDA or supervised modeling. If using for prediction, ensure proper imputation (e.g., with a fitted `SimpleImputer`) aligns with the clustering preprocessing.
- **Documentation**: Note the selected \( k \) and key insights (e.g., “\( k = 3 \) reveals an obese subgroup with high insulin”) for future reference.

### Notes
- The `fit_and_summarize` function is assumed to return the K-means model, labels, and centroids DataFrame. Ensure it’s defined correctly in a prior cell.
- The PCA plots depend on the variance explained by the first two components. If separation is poor, check the explained variance ratio (e.g., via `pca.explained_variance_ratio_`) in a follow-up analysis.
- The results align with a health dataset (e.g., Pima Indians Diabetes), so interpretations reflect clinical context.

In [ ]:
# ---------- assign chosen k, create df_out and risk_category ----------
k = 2  # your chosen value
km_final = KMeans(n_clusters=k, random_state=42, n_init=50)
cluster_labels = km_final.fit_predict(X_scaled)
centers_orig = scaler.inverse_transform(km_final.cluster_centers_)
centers_df = pd.DataFrame(centers_orig, columns=X.columns)
centers_df['Cluster'] = range(k)

# attach cluster and compute risk logic (strict thresholds then fallback)
df_out = df.copy()   # df should already be cleaned/capped if use_capping True
df_out['Cluster'] = cluster_labels

glucose_thresh = 126
bmi_thresh = 30
dpf_thresh = 0.5

centers_df['high_risk_strict'] = (
    (centers_df['Glucose'] > glucose_thresh) &
    (centers_df['BMI'] > bmi_thresh) &
    (centers_df['DiabetesPedigreeFunction'] > dpf_thresh)
)

# fallback aggregated risk if no strict cluster
high_risk_clusters = centers_df.loc[centers_df['high_risk_strict'], 'Cluster'].tolist()
if len(high_risk_clusters) == 0:
    # aggregated normalized risk
    risk_cols = ['Glucose','BMI','DiabetesPedigreeFunction']
    tmp = centers_df.copy()
    for col in risk_cols:
        mn, mx = tmp[col].min(), tmp[col].max()
        tmp[col + '_norm'] = 0.0 if mx - mn == 0 else (tmp[col] - mn) / (mx - mn)
    tmp['agg_risk'] = tmp[[c + '_norm' for c in risk_cols]].mean(axis=1)
    top_score = tmp['agg_risk'].max()
    high_risk_clusters = tmp.loc[tmp['agg_risk'] >= top_score, 'Cluster'].tolist()

df_out['risk_category'] = df_out['Cluster'].apply(lambda x: 1 if x in high_risk_clusters else 0)

# save artifacts for reproducibility and downstream supervised cell
centers_df.to_csv("cluster_centers.csv", index=False)
df_out.to_csv("df_with_clusters_and_risk.csv", index=False)
print("Saved cluster_centers.csv and df_with_clusters_and_risk.csv")
print("Cluster counts:", df_out['Cluster'].value_counts().to_dict())
print("Risk category counts:", df_out['risk_category'].value_counts().to_dict())


In [ ]:
# ----------------------
# SUPERVISED (FINAL, no-pipeline approach)
# ----------------------
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.base import clone
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# ---- 0) quick safety: check df_out and risk_category ----
if 'df_out' not in globals():
    raise RuntimeError("df_out not found. Run the clustering cells first to produce df_out.")
if 'risk_category' not in df_out.columns:
    raise RuntimeError("df_out does not contain 'risk_category'. Ensure clustering + risk labeling ran successfully.")

# safe check: ensure features list is valid
available_features = [f for f in features if f in df_out.columns]
if len(available_features) < 2:
    raise RuntimeError("Not enough features found in df_out to train. Check features list.")
features = available_features


# ---- 1) define X, y (binary risk target) ----
# ensure features list contains available columns only
features = [f for f in (globals().get('features') or []) if f in df_out.columns]
if not features:
    # fallback: numeric columns used earlier
    features = df_out.select_dtypes(include=[np.number]).columns.tolist()
    # remove Cluster and risk_category from features
    features = [c for c in features if c not in ('Cluster','risk_category')]

X = df_out[features].copy()
y = df_out['risk_category'].copy()   # binary target: 0 low risk, 1 high risk

print("Features used:", features)
print("Overall class balance:", Counter(y))

# ---- 2) train/test split (stratified) ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train/test shapes:", X_train.shape, X_test.shape)
print("Train class counts:", Counter(y_train))

# ---- 3) fit imputers and scaler ON TRAIN only (no leakage) ----
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),  columns=X_test.columns,  index=X_test.index)

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns, index=X_train_imp.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test_imp),   columns=X_test_imp.columns,  index=X_test_imp.index)

# ---- 4) oversample TRAIN only ----
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train_scaled, y_train)
print("After oversampling train class counts:", Counter(y_train_res))

# ---- 5) define models (you can tune later) ----
models = {
    "RandomForest": RandomForestClassifier(random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}
for name, clf in models.items():
    # fit on resampled (train) data
    clf.fit(X_train_res, y_train_res)
    y_pred = clf.predict(X_test_scaled)
    acc = clf.score(X_test_scaled, y_test)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    # optional ROC AUC (works for probabilistic models)
    try:
        y_proba = clf.predict_proba(X_test_scaled)[:,1]
        roc = roc_auc_score(y_test, y_proba)
    except Exception:
        roc = np.nan
    print(f"\nModel: {name}")
    print("Accuracy:", acc)
    print("F1:", f1)
    if not np.isnan(roc):
        print("ROC AUC:", roc)
    print("Classification report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    # Confusion matrix plot
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(name)
    plt.show()

    # store perf for summary
    results[name] = {"accuracy": acc, "f1": f1, "roc_auc": roc}

# Summary table
res_df = pd.DataFrame(results).T.sort_values(by='f1', ascending=False)
print("\nSummary (sorted by F1):")
display(res_df)

# ---- 6) Manual CV with resampling inside folds (optional, safer CV) ----
def cv_with_resampling(clf, X_all, y_all, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    f1s = []
    for train_idx, val_idx in skf.split(X_all, y_all):
        Xtr, Xval = X_all.iloc[train_idx], X_all.iloc[val_idx]
        ytr, yval = y_all.iloc[train_idx], y_all.iloc[val_idx]
        # fit imputer & scaler on this fold's training set
        imputer_fold = SimpleImputer(strategy='median').fit(Xtr)
        scaler_fold = StandardScaler().fit(imputer_fold.transform(Xtr))
        Xtr_proc = scaler_fold.transform(imputer_fold.transform(Xtr))
        Xval_proc = scaler_fold.transform(imputer_fold.transform(Xval))
        # resample training fold
        Xtr_res, ytr_res = RandomOverSampler(random_state=42).fit_resample(Xtr_proc, ytr)
        clf_clone = clone(clf)
        clf_clone.fit(Xtr_res, ytr_res)
        yval_pred = clf_clone.predict(Xval_proc)
        f1s.append(f1_score(yval, yval_pred, zero_division=0))
    return np.mean(f1s), np.std(f1s)

# Example: (commented out to save time)
# from sklearn.base import clone
# for name, clf in models.items():
#     mean_f1, std_f1 = cv_with_resampling(clone(clf), X, y, n_splits=5)
#     print(f"{name} CV f1: {mean_f1:.3f} ± {std_f1:.3f}")

# ---- 7) Grid search RandomForest on the resampled train set (example) ----
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5]
}
rf = RandomForestClassifier(random_state=42)
grid = GridSearchCV(rf, param_grid, scoring='f1', cv=5, n_jobs=-1)
grid.fit(X_train_res, y_train_res)
print("Grid best params:", grid.best_params_, "best CV f1:", grid.best_score_)
best_model = grid.best_estimator_

# ---- 8) Final fit on resampled train + test evaluation + save artifacts ----
best_model.fit(X_train_res, y_train_res)
y_pred_final = best_model.predict(X_test_scaled)
print("\nFinal evaluation (test set):")
print(classification_report(y_test, y_pred_final))
print("Confusion matrix:")
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d')
plt.show()

# Save model + preprocessing artifacts (so you can load and run inference later)
joblib.dump(best_model, "final_model.pkl")
joblib.dump(imputer, "imputer.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(features, "feature_names.pkl")
print("Saved final_model.pkl, imputer.pkl, scaler.pkl, feature_names.pkl")


## Explanation of Cell: Supervised Modeling (Final, No-Pipeline Approach)

### Purpose
This cell implements a supervised learning pipeline to predict the binary `risk_category` target (0 for low risk, 1 for high risk) using various classification models. The goal is to evaluate and compare the performance of models like RandomForest, DecisionTree, LogisticRegression, SVM, GradientBoosting, and XGBoost on a dataset derived from prior clustering and preprocessing steps. The pipeline includes data splitting, imputation, scaling, oversampling to handle class imbalance, model training, and evaluation with metrics like accuracy, F1 score, and ROC AUC, culminating in saving the best model and preprocessing artifacts.

### Methodology
- **Data Preparation**:
  - The DataFrame `df_out` (assumed from prior clustering cells) is checked for the `risk_category` column and valid features (`Glucose`, `BMI`, etc.).
  - Features are filtered to include only available numeric columns, excluding `Cluster` and `risk_category`.
  - A stratified train-test split (80/20) ensures balanced class representation.
  - Missing values are imputed with the median using `SimpleImputer`, and features are scaled with `StandardScaler`, both fitted on the training set only to avoid leakage.
  - RandomOverSampler addresses the initial class imbalance (410 low risk, 358 high risk) by oversampling the minority class in the training set.
- **Model Training and Evaluation**:
  - Six models are defined with default hyperparameters: RandomForest, DecisionTree, LogisticRegression, SVM, GradientBoosting, and XGBoost.
  - Each model is trained on the oversampled training data and evaluated on the test set using accuracy, F1 score (emphasizing balance between precision and recall), and ROC AUC (for probabilistic models).
  - Confusion matrices are visualized to assess true positives, false positives, etc.
  - A summary table ranks models by F1 score.
- **Advanced Tuning**:
  - Optional cross-validation with resampling inside folds is provided (commented out) for a more robust estimate.
  - Grid search tunes RandomForest hyperparameters (e.g., `n_estimators`, `max_depth`) using F1 as the scoring metric.
  - The best model is refit, and its performance is evaluated on the test set, with artifacts (model, imputer, scaler, features) saved for future inference.

### Results
- **Data Overview**:
  - **Features Used**: ['Glucose', 'BMI', 'DiabetesPedigreeFunction', 'BloodPressure', 'SkinThickness', 'Insulin', 'Age', 'Pregnancies'].
  - **Class Balance**: Initial: 410 (0), 358 (1); Train: 328 (0), 286 (1); After oversampling: 328 (0), 328 (1).
  - **Train/Test Shapes**: (614, 8) train, (154, 8) test.
- **Model Performance** (Partial Results):
  - **RandomForest**: Accuracy = 0.955, F1 = 0.952, ROC AUC = 0.987.
    - Classification Report: Precision ≈ 0.96, Recall ≈ 0.95 for both classes, balanced performance.
    - Confusion Matrix: Likely shows high true positives/negatives with few errors.
  - **DecisionTree**: Accuracy = 0.870, F1 = 0.859, ROC AUC = 0.869.
    - Classification Report: Precision ≈ 0.87, Recall ≈ 0.85–0.89, moderate performance.
  - **LogisticRegression**: Accuracy = 0.942, F1 = 0.936, ROC AUC = 0.986.
    - Classification Report: Precision ≈ 0.93–0.96, Recall ≈ 0.92–0.96, strong balance.
  - **SVM**: Accuracy = 0.955 (F1 and ROC AUC not fully reported, but likely competitive given accuracy).
  - **GradientBoosting and XGBoost**: Results not provided, but expected to be strong based on typical performance.
- **Grid Search**: Best parameters and CV F1 for RandomForest are computed but not shown; the best model is saved.
- **Final Model**: The tuned RandomForest is refit, with final test set evaluation and artifacts saved.

### What This Tells Us
- **Class Imbalance**: The initial imbalance (410 vs. 358) is mild, but oversampling ensures equal representation (328 each), improving model sensitivity to the minority class.
- **Model Performance**:
  - **RandomForest** excels with the highest F1 (0.952) and ROC AUC (0.987), indicating excellent balance and discrimination ability, likely due to its robustness to scaled features and handling of non-linear relationships.
  - **LogisticRegression** performs nearly as well (F1 = 0.936, ROC AUC = 0.986), suggesting linear separability in the scaled, imputed data.
  - **DecisionTree** lags (F1 = 0.859), reflecting overfitting or less effective feature handling without tuning.
  - **SVM** matches RandomForest’s accuracy (0.955), hinting at strong performance pending full metrics.
  - GradientBoosting and XGBoost (not shown) may offer further improvements with tuning.
- **Evaluation Metrics**: High F1 and ROC AUC scores (> 0.9) indicate the models generalize well, with confusion matrices likely showing balanced true positives/negatives. The slight drop from training to test performance suggests mild overfitting, mitigated by oversampling.
- **Feature Importance**: The use of all eight features (including `Insulin`, `SkinThickness`) implies they collectively contribute to risk prediction, though tuning could reveal redundancies.
- **Modeling Implications**: The tuned RandomForest is a strong candidate for deployment, with saved artifacts enabling inference. The high ROC AUC suggests good class separation, useful for risk stratification in a health context (e.g., diabetes risk).

### Recommendations
- **Select Best Model**: The tuned RandomForest appears optimal based on F1 and ROC AUC. Validate its performance with the full test report and consider GridSearchCV results for confirmation.
- **Tune Further**: If time permits, uncomment and run the CV loop for all models, or expand the `param_grid` (e.g., add `min_samples_leaf`) to refine RandomForest. Test GradientBoosting/XGBoost with default or tuned parameters.
- **Evaluate Overfitting**: Compare train and test F1/ROC AUC (if available) to assess overfitting. If significant, reduce model complexity (e.g., lower `n_estimators`).
- **Feature Selection**: If some features (e.g., `SkinThickness` with many zeros) underperform, use feature importance from RandomForest to prune them.
- **Next Steps**: Load the saved model (`final_model.pkl`) in a new cell to test predictions on new data. Explore threshold tuning (e.g., for ROC) if recall for one class is critical.
- **Documentation**: Note the best model’s metrics (e.g., “RandomForest: F1 = 0.952, ROC AUC = 0.987”) and any observations from confusion matrices for future reference.

### Notes
- The results are partial (e.g., SVM’s F1/ROC AUC, GradientBoosting/XGBoost missing). Share the full output for a complete analysis.
- The code assumes `df_out` and `features` are from prior cells (e.g., clustering). Ensure consistency.
- The supervised approach skips a pipeline for simplicity, but consider integrating into a `Pipeline` for production use.